# RichterPy Pipeline Notebook

`RICHTER.md` is the ground truth for pipeline names. This notebook documents the effective Python APIs currently used by each implemented stage.

- `A1 & A2 -> O1 -> O`: raw `.srm/.wve` streams plus `sensorarray/EXPERIMENT.csv` station metadata into ObsPy.
- `A2 & B1 -> O2 -> O`: raw `.srm/.wve` streams plus `EXPERIMENT.pcf` station metadata into ObsPy.
- `B1 & C1 -> O3 -> O`: triggered BSF/ESF/ATF streams plus PCF-derived station metadata into ObsPy.
- `C1 -> O4 -> O`: ESF-native event metadata into ObsPy `Catalog` / QuakeML.
- `O -> P`: optional Pyrocko/Snuffler visualization.

The paths below target the local MEERA `m0013` data layout. Change them for another experiment.

## Shared Configuration

These paths are reused by the stage examples. `DATUM_CONFIG` supplies the MEERA datum for converting local `North/East/Down` coordinates to geographic coordinates.

In [1]:
EXPERIMENT = "m0013"
DATUM_CONFIG = "config/MEERA.ini"

RAW_STREAM_PATH = r"\\hphtfear\hphtfear\m0013"
CSV_STATIONS_PATH = r"\\hphtfear\hphtfear\00_PROJECTS\m0013\sensorarray\m0013.csv"
PCF_PATH = r"\\hphtfear\hphtfear\00_PROJECTS\m0013\m0013.pcf"

TRIGGERED_EVENTS_BSF_PATH = r"\\hphtfear\hphtfear\00_PROJECTS\m0013\BSF"
TRIGGERED_EVENTS_ESF_PATH = r"\\hphtfear\hphtfear\00_PROJECTS\m0013\ESF"
TRIGGERED_EVENTS_ESF_FILE = r"\\hphtfear\hphtfear\00_PROJECTS\m0013\ESF\20250403\20250403_0429.ESF"
TRIGGERED_EVENTS_ATF_PATH = TRIGGERED_EVENTS_ESF_PATH

CSV_STATIONS_XML = "data/playground/m0013.stations.csv.xml"
PCF_STATIONS_CSV = "data/playground/m0013.pcf.csv"
PCF_STATIONS_XML = "data/playground/m0013.stations.pcf.xml"
ESF_QUAKEML = "data/playground/m0013.events.esf.xml"

## A1 & A2 -> O1 -> O

Raw `.srm/.wve` streams plus station CSV metadata into ObsPy.

Effective APIs: `richterpy.convert.stations.convert_stations(...)` and `richterpy.convert.snuffler.build_master_stream(...)`.

In [2]:
from richterpy.convert.stations import convert_stations
from richterpy.convert.snuffler import build_master_stream, build_station_channel_map
import obspy

convert_stations(csv_path=CSV_STATIONS_PATH, output_path=CSV_STATIONS_XML, datum_config=DATUM_CONFIG)
inv = obspy.read_inventory(CSV_STATIONS_XML)
station_channel_map = build_station_channel_map(inv)
st = build_master_stream(RAW_STREAM_PATH, station_channel_map=station_channel_map)

--> Loading data from \\hphtfear\hphtfear\00_PROJECTS\m0013\sensorarray\m0013.csv...
--> StationXML for m0013 written to data\playground\m0013.stations.csv.xml.

--- 🔍 Searching for Data Waveforms ---
Found 7 SRM files. Assembling master stream...
Loading: richter-m.m0013.00001.srm | Start: 2025-04-03T12:18:19.009000Z
Loading: richter-m.m0013.00002.srm | Start: 2025-04-03T12:28:19.189000Z
Loading: richter-m.m0013.00003.srm | Start: 2025-04-03T12:38:19.380000Z
Loading: richter-m.m0013.00004.srm | Start: 2025-04-03T12:48:19.556000Z
Loading: richter-m.m0013.00005.srm | Start: 2025-04-03T12:58:19.747000Z
Loading: richter-m.m0013.00006.srm | Start: 2025-04-03T13:08:19.938000Z
Loading: richter-m.m0013.00007.srm | Start: 2025-04-03T13:18:20.112000Z


## A2 & B1 -> O2 -> O

Raw `.srm/.wve` streams plus PCF-derived station metadata into ObsPy.

Effective APIs: `richterpy.io.pcf.convert_pcf_to_csv(...)`, `convert_stations(...)`, and `build_master_stream(...)`.

In [3]:
from richterpy.io.pcf import convert_pcf_to_csv
from richterpy.convert.stations import convert_stations
from richterpy.convert.snuffler import build_master_stream, build_station_channel_map
import obspy

convert_pcf_to_csv(PCF_PATH, output_path=PCF_STATIONS_CSV)
convert_stations(csv_path=PCF_STATIONS_CSV, output_path=PCF_STATIONS_XML, datum_config=DATUM_CONFIG)
inv = obspy.read_inventory(PCF_STATIONS_XML)
station_channel_map = build_station_channel_map(inv)
st = build_master_stream(RAW_STREAM_PATH, station_channel_map=station_channel_map)

--> Loading data from data\playground\m0013.pcf.csv...
--> StationXML for m0013.pcf written to data\playground\m0013.stations.pcf.xml.

--- 🔍 Searching for Data Waveforms ---
Found 7 SRM files. Assembling master stream...
Loading: richter-m.m0013.00001.srm | Start: 2025-04-03T12:18:19.009000Z
Loading: richter-m.m0013.00002.srm | Start: 2025-04-03T12:28:19.189000Z
Loading: richter-m.m0013.00003.srm | Start: 2025-04-03T12:38:19.380000Z
Loading: richter-m.m0013.00004.srm | Start: 2025-04-03T12:48:19.556000Z
Loading: richter-m.m0013.00005.srm | Start: 2025-04-03T12:58:19.747000Z
Loading: richter-m.m0013.00006.srm | Start: 2025-04-03T13:08:19.938000Z
Loading: richter-m.m0013.00007.srm | Start: 2025-04-03T13:18:20.112000Z


## B1 & C1 -> O3 -> O

Triggered BSF/ESF/ATF streams plus PCF-derived station metadata into ObsPy.

Effective APIs: `build_bsf_stream(...)`, `build_esf_stream(...)`, or `build_atf_stream(...)`. The station inventory still comes from `B1` via `convert_pcf_to_csv(...)` and `convert_stations(...)`.

In [4]:
# Choose one of: "BSF", "ESF", "ATF".
triggered = "ATF"

match triggered:
    case "BSF":
        from richterpy.io.bsf import build_bsf_stream
        st = build_bsf_stream(TRIGGERED_EVENTS_BSF_PATH, station_channel_map=station_channel_map)
    case "ESF":
        from richterpy.io.esf import build_esf_stream
        st = build_esf_stream(TRIGGERED_EVENTS_ESF_PATH, station_channel_map=station_channel_map)
    case "ATF":
        from richterpy.io.atf import build_atf_stream
        st = build_atf_stream(TRIGGERED_EVENTS_ATF_PATH, station_channel_map=station_channel_map)

print(st)

20 Trace(s) in Stream:
RC.S01.RAW.NDZ | 2025-04-03T14:37:09.847501Z - 2025-04-03T14:37:09.854055Z | 10000000.0 Hz, 65536 samples
RC.S02.RAW.NDZ | 2025-04-03T14:37:09.847501Z - 2025-04-03T14:37:09.854055Z | 10000000.0 Hz, 65536 samples
RC.S03.RAW.NDZ | 2025-04-03T14:37:09.847501Z - 2025-04-03T14:37:09.854055Z | 10000000.0 Hz, 65536 samples
RC.S04.RAW.ND1 | 2025-04-03T14:37:09.847501Z - 2025-04-03T14:37:09.854055Z | 10000000.0 Hz, 65536 samples
RC.S01.RAW.NDZ | 2025-04-03T14:51:20.653475Z - 2025-04-03T14:51:20.660029Z | 10000000.0 Hz, 65536 samples
RC.S02.RAW.NDZ | 2025-04-03T14:51:20.653475Z - 2025-04-03T14:51:20.660029Z | 10000000.0 Hz, 65536 samples
RC.S03.RAW.NDZ | 2025-04-03T14:51:20.653475Z - 2025-04-03T14:51:20.660029Z | 10000000.0 Hz, 65536 samples
RC.S04.RAW.ND1 | 2025-04-03T14:51:20.653475Z - 2025-04-03T14:51:20.660029Z | 10000000.0 Hz, 65536 samples
RC.S01.RAW.NDZ | 2025-04-03T14:51:20.658391Z - 2025-04-03T14:51:20.664944Z | 10000000.0 Hz, 65536 samples
RC.S02.RAW.NDZ | 2025-0

## C1 -> O4 -> O

ESF-native event metadata into ObsPy `Catalog` / QuakeML.

Effective API: `richterpy.io.esf.build_esf_catalog_from_esfs(...)`. Pass the PCF-derived StationXML so ESF picks get real waveform IDs. CLI equivalent: `richter-esf-quakeml <ESF-file-or-directory> --datum-config config/MEERA.ini --station-xml-path <station.xml>`.

In [ ]:
# ESF -> ObsPy Catalog / QuakeML test cell.
# Use TRIGGERED_EVENTS_ESF_FILE for a fast smoke test, or TRIGGERED_EVENTS_ESF_PATH for the full ESF directory.
from richterpy.io.esf import build_esf_catalog_from_esfs

cat = build_esf_catalog_from_esfs(
    TRIGGERED_EVENTS_ESF_FILE,
    output_path=ESF_QUAKEML,
    datum_config=DATUM_CONFIG,
    inventory_path=PCF_STATIONS_XML,
)

917 Event(s) in Catalog:
2025-04-03T14:28:40.380641Z | +41.828,  +12.515 | -3.01 Local Magnitude
2025-04-03T14:28:52.209319Z | +41.828,  +12.515 | -3.16 Local Magnitude
...
2025-04-03T15:27:28.027964Z | +41.828,  +12.515
2025-04-03T15:27:28.033231Z | +41.828,  +12.515 | -2.70 Local Magnitude
To see all events call 'print(CatalogObject.__str__(print_all=True))'


In [7]:
print(cat)

917 Event(s) in Catalog:
2025-04-03T14:28:40.380641Z | +41.828,  +12.515 | -3.01 Local Magnitude
2025-04-03T14:28:52.209319Z | +41.828,  +12.515 | -3.16 Local Magnitude
...
2025-04-03T15:27:28.027964Z | +41.828,  +12.515
2025-04-03T15:27:28.033231Z | +41.828,  +12.515 | -2.70 Local Magnitude
To see all events call 'print(CatalogObject.__str__(print_all=True))'


In [11]:
cat[0].picks[0].comments

[Comment(text='p_timepick_sample_1based: 14961', resource_id=ResourceIdentifier(id="smi:local/89edf586-2a03-4636-aa52-b48ced02f8bc"))]

## O -> P

ObsPy to Pyrocko/Snuffler. This section is interactive and opens a GUI window. Run it only in a graphical session.
NB all zooming problems in Pyrocko disappear when it is launched from WSL.

In [12]:
%matplotlib qt

In [13]:
from pyrocko import obspy_compat, trace
from pyrocko.gui.snuffler import marker
from richterpy.convert.snuffler import get_high_precision_events

obspy_compat.plant()

# Convert ObsPy event origins and ESF-derived picks to Pyrocko/Snuffler objects.
# The catalog must be built with inventory_path=PCF_STATIONS_XML so picks have real waveform IDs.
pyrocko_events = get_high_precision_events(cat)

pick_markers = []
for event in cat:
    for pick in event.picks:
        waveform_id = pick.waveform_id
        if waveform_id is None:
            continue
        nslc = (
            waveform_id.network_code or "",
            waveform_id.station_code or "",
            waveform_id.location_code or "",
            waveform_id.channel_code or "",
        )
        pick_markers.append(
            marker.PhaseMarker(
                nslc_ids=[nslc],
                tmin=pick.time.timestamp,
                tmax=pick.time.timestamp,
                phasename=pick.phase_hint or "P",
            )
        )

trace.snuffle(
    st.to_pyrocko_traces(),
    stations=inv.to_pyrocko_stations(),
    events=pyrocko_events,
    markers=pick_markers,
)


HPFloatUnavailable: NumPy lacks support for float128 or float96 data type on this platform.

Blocking reentrant call to paintEvent().
Blocking reentrant call to paintEvent().
Blocking reentrant call to paintEvent().
